# Assessment 1: Analysing historical data with system performance - Phase 2

**Student ID:** 35721588  
**Unit:** ITO5202  
**Teaching Period:** 5, 2026

**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Source:** https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

---

## Contents

**Part A: Analytical query design and implementation**
1. Business query design and justification
2. DataFrame API implementation
3. Spark SQL implementation
4. Result validation and API comparison

**Part B: System perspective and performance analysis**
1. Partitioning strategy analysis
2. Execution time benchmarking
3. Execution plan interpretation
4. DAG analysis via the Spark Web UI

---

## Environment and configuration

### Execution environment

For this project, we plan to run Spark in **local mode** on a single machine. In this setup, Spark does not create separate executor JVMs. Instead, the driver process handles the computation itself and uses the machine’s available logical CPU cores to run tasks in parallel. Because of this, the main memory setting that matters for our setup is spark.driver.memory, so we do not need to configure executor memory separately.

| Property | Value |
|---|---|
| Machine | MacBook Air (Retina, 13-inch, 2018) |
| Processor | 1.6 GHz dual-core Intel Core i5 |
| Logical cores | 4 |
| Physical memory | 8 GB |
| Operating system | macOS Sonoma 14.7.8 |
| Java | Eclipse Temurin JDK 17 (x64) |
| Python | 3.11.9 |
| PySpark | 3.5.1 |
| Spark master | `local[*]` |

The environment details are generated directly in the notebook rather than written in manually. This means the values referred to later in the Part B benchmarking discussion can be checked against the notebook output.

In [1]:
# Import packages
import os
import time
from statistics import median
import pandas as pd

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Set up data directory folder path
DATA_DIR = "data"

## SparkSession configuration

For this project, there are three key Spark settings which we changed from their default values due to how they affect the behaviour we want to examine later in Part B.

**`spark.driver.memory = 3g`.** Our machine has 8 GB of RAM, which also needs to support the computer's other processes. Allocating too much memory to Spark could slow  down our execution significantly. This matters to us beyond a speed perspective, since Part B.2 compares execution times. More specifically, this would make our results less useful because they could reflect memory pressure rather than Spark's actual processing behaviour.

**`spark.sql.shuffle.partitions = 4`.** This setting determines how many partitions Spark creates after a shuffle. The default is 200, which makes more sense for a much larger cluster than for the local environment we are using here. With four available task slots and a dataset of around 113,000 rows at its largest, using 200 partitions would create many very small tasks and add unnecessary scheduling overhead.

**`spark.sql.adaptive.enabled = false`.** Spark's Adaptive Query Execution (AQE) can change the physical execution plan while a query is running. On one hand, this can improve performance, but on the other hand, it makes the execution harder to compare with the plan shown by `explain(extended=True)`. Because Parts B.3 and B.4 require us to examine the physical plan and its corresponding DAG, AQE is turned off so that the printed plan and the executed plan remain consistent. This also means that the partition counts discussed in Part B.1 reflect the values we set orselves rather than values Spark changes during execution.


In [2]:
# Spark session build with local mode, 4 task slots, AQE off (as explained above)
spark = SparkSession.builder \
    .appName("ITO5202-A1-Olist-Freight") \
    .master("local[*]") \
    .config("spark.driver.memory", "3g") \
    .config("spark.sql.shuffle.partitions", 4) \
    .config("spark.sql.adaptive.enabled", False) \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark

26/09/20 01:00:40 WARN Utils: Your hostname, Mounishas-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.137 instead (on interface en0)
26/09/20 01:00:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/20 01:00:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
sc = spark.sparkContext

# Print the main Spark environment settings
print("Spark version:", spark.version)
print("Master:", sc.master)
print("Available task slots:", sc.defaultParallelism)
print("Driver memory:", spark.conf.get("spark.driver.memory"))
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Broadcast join threshold (bytes):", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

# Web UI link
print("Web UI:", sc.uiWebUrl)

Spark version: 3.5.1
Master: local[*]
Available task slots: 4
Driver memory: 3g
Shuffle partitions: 4
AQE enabled: false
Broadcast join threshold (bytes): 10485760b
Web UI: http://192.168.0.137:4040


In [4]:
# Confirms the session can run a job end to end
spark.range(10).count()

10

## Data Loading

### Defining schemas

Our nine source files are loaded in using manually defined schemas instead of `inferSchema=True`. This is because by us using schema inference, Spark has to inspect the data before it is able to load it, thus adding extra work (this is particularly important for the geolocation file, which contains over one million rows).

Furthermore, if we were to allow Spark to infer the schema automatically, the same column in different data files could be interpreted as being of different data types, which can have downstream impacts on analysis when we try to do joins. Instead, by defining the schema ourselves, we are able to ensure consistency in data types. 

We also note that as per our proposal, we are looking to only use seven of the nine available data files, as do not wish to include the payments and reviews dataset as they fall outside the scope of our analysis. As such, we do not read in these datasets such as to avoid unneccessary extra work.

### NOTE

Given a heading error in Section 3 of our original proposal, we need to make a correction in our dataset list. The first table in Section 3 of our approved proposal is labelled `olist_geolocation_dataset.csv`, but the columns listed underneath it actually belong to `olist_order_items_dataset.csv`. The geolocation dataset then appears again later in the sme section of our proposal with the correct columns.

We note that the approved proposal has been kept unchanged in `proposal/proposal.md` for consistency. However, the schemas defined below use the actual, correct columns from each source file, which were checked against the downloaded dataset.
